# Atelier Préparation de Données Images 

## Partie 1 – Exploration du dataset 

Développer un programme Python capable de récupérer, pour chaque image, son nom, sa classe, son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa taille. 

NB : prendre en charge aussi les fichiers corrompus 

### Cellule de code 1 imports

In [1]:
import os                      # pour parcourir les dossiers et récupérer la taille des fichiers
import numpy as np             # pour calculer l'écart-type des pixels
from PIL import Image          # pour ouvrir et lire les métadonnées des images
import pandas as pd            # pour stocker les résultats dans un tableau (DataFrame)

RAW_DIR = "../data/raw"        # chemin vers le dossier contenant les images brutes, par classe

### Cellule de code 2 fonction d'extraction pour une image

In [2]:
def extraire_infos_image(chemin_fichier, classe):
    # chemin_fichier : chemin complet vers l'image à analyser
    # classe : nom du sous-dossier (cardboard, glass, metal, paper, plastic, trash)

    infos = {
        "nom": os.path.basename(chemin_fichier),   # nom du fichier seul (sans le chemin)
        "classe": classe,                          # classe déduite du dossier parent
        "format": None,                            # sera rempli si l'image s'ouvre correctement
        "mode": None,
        "largeur": None,
        "hauteur": None,
        "ecart_type_pixels": None,
        "nb_canaux": None,
        "taille_octets": os.path.getsize(chemin_fichier),  # taille du fichier sur le disque
        "corrompue": False,                         # drapeau : True si l'ouverture échoue
    }

    try:
        with Image.open(chemin_fichier) as img:     # tentative d'ouverture de l'image
            img.verify()                            # vérifie l'intégrité du fichier (détecte la corruption)
        # img.verify() rend l'objet inutilisable pour la suite, donc on rouvre l'image
        with Image.open(chemin_fichier) as img:
            infos["format"] = img.format            # ex: JPEG, PNG
            infos["mode"] = img.mode                 # ex: RGB, L (grayscale), RGBA
            infos["largeur"], infos["hauteur"] = img.size  # (largeur, hauteur) en pixels

            tableau_pixels = np.array(img)           # conversion de l'image en tableau numpy
            infos["ecart_type_pixels"] = tableau_pixels.std()  # écart-type, tous canaux aplatis automatiquement

            # nombre de canaux : 1 si l'image est 2D (grayscale), sinon la 3e dimension du tableau
            infos["nb_canaux"] = 1 if tableau_pixels.ndim == 2 else tableau_pixels.shape[2]

    except Exception:
        # si l'ouverture ou la lecture échoue, on marque l'image comme corrompue
        # et on laisse les autres champs à None plutôt que de faire planter le programme
        infos["corrompue"] = True

    return infos

### Cellule de code 3 parcours du dataset et construction du DataFrame

In [4]:
EXTENSIONS_VALIDES = (".jpg", ".jpeg", ".png")  # extensions d'images qu'on accepte

resultats = []  # liste qui va accueillir un dictionnaire d'infos par image

# on récupère la liste des classes = noms des sous-dossiers de RAW_DIR
classes = sorted(os.listdir(RAW_DIR))

for classe in classes:
    dossier_classe = os.path.join(RAW_DIR, classe)   # ex: ../data/raw/cardboard

    if not os.path.isdir(dossier_classe):
        continue  # on ignore les éléments qui ne sont pas des dossiers

    for nom_fichier in os.listdir(dossier_classe):
        chemin_fichier = os.path.join(dossier_classe, nom_fichier)  # chemin complet vers le fichier

        if not os.path.isfile(chemin_fichier):
            continue  # on ignore les sous-dossiers éventuels

        if not nom_fichier.lower().endswith(EXTENSIONS_VALIDES):
            continue  # on ignore les fichiers qui ne sont pas des images (ex: Zone.Identifier)

        infos = extraire_infos_image(chemin_fichier, classe)  # appel de la fonction de la cellule 2
        resultats.append(infos)  # ajout du dictionnaire à la liste globale

# conversion de la liste de dictionnaires en tableau pandas : une ligne = une image
df_audit = pd.DataFrame(resultats)

print(f"Nombre total d'images analysées : {len(df_audit)}")
df_audit.head()  # aperçu des premières lignes du tableau

Nombre total d'images analysées : 1030


,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue
0,cardboard95.jpg,cardboard,JPEG,RGB,512.0,384.0,75.067740,3.0,30705,False
1,cardboard51.jpg,cardboard,JPEG,RGB,512.0,384.0,45.727011,3.0,21683,False
2,cardboard111.jpg,cardboard,JPEG,RGB,512.0,384.0,45.872520,3.0,24881,False
3,cardboard24.jpg,cardboard,JPEG,RGB,512.0,384.0,42.975044,3.0,19051,False
4,cardboard98.jpg,cardboard,JPEG,RGB,512.0,384.0,56.502087,3.0,21538,False


In [5]:
df_audit["classe"].value_counts()

classe
paper        252
plastic      224
glass        187
cardboard    168
metal        149
trash         50
Name: count, dtype: int64

## Partie 2 – Détecter les images corrompues

Écrire et se servir d’une fonction qui détecte une image corrompue. 

In [7]:
def est_corrompue(chemin_fichier):
    # tente d'ouvrir et de vérifier l'intégrité du fichier
    # renvoie True si le fichier est corrompu, False s'il est valide
    try:
        with Image.open(chemin_fichier) as img:
            img.verify()
        return False
    except Exception:
        return True


# on reparcourt le dataset en appliquant uniquement cette fonction,
# en réutilisant le même filtre d'extension que la Partie 1
images_corrompues = []  # chemins des images détectées comme corrompues

for classe in classes:  # variable déjà définie à la Partie 1
    dossier_classe = os.path.join(RAW_DIR, classe)

    if not os.path.isdir(dossier_classe):
        continue

    for nom_fichier in os.listdir(dossier_classe):
        chemin_fichier = os.path.join(dossier_classe, nom_fichier)

        if not os.path.isfile(chemin_fichier):
            continue

        if not nom_fichier.lower().endswith(EXTENSIONS_VALIDES):
            continue  # on ignore les fichiers non-images (ex: Zone.Identifier)

        if est_corrompue(chemin_fichier):
            images_corrompues.append(chemin_fichier)

print(f"Nombre d'images corrompues détectées : {len(images_corrompues)}")
images_corrompues

Nombre d'images corrompues détectées : 6


['../data/raw/cardboard/cardboard83.jpg',
 '../data/raw/glass/glass74.jpg',
 '../data/raw/metal/metal48.jpg',
 '../data/raw/paper/paper213.jpg',
 '../data/raw/plastic/plastic13.jpg',
 '../data/raw/trash/trash3.jpg']